In [1]:
import pandas as pd
import numpy as np
import itertools
import datetime
import pandas_gbq
import matplotlib.pyplot as plt
from datetime import *
from datetime import datetime, timedelta, date
# %load_ext google.colab.data_table
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
project_id = "perceptive-ivy-290216"

# Standard plotly imports
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
query2=f"""
SELECT * FROM `perceptive-ivy-290216.f1_api.sprint_qualifying_lap_time`
# WHERE YEAR=2023
# AND GP='Spanish Grand Prix'
"""
track3=pd.read_gbq(query2,project_id,dialect='standard')

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_37839/1060962308.py:6: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  track3=pd.read_gbq(query2,project_id,dialect='standard')


In [6]:
track2=track3[(track3["GP"]=='Qatar Grand Prix')&(track3["Year"]==2025)]
year=track2['Year'].iloc[0]
gp=track2['GP'].iloc[0]

In [7]:
Drivers=(track2['Driver'].unique())
Drivers

array(['VER', 'NOR', 'BOR', 'HAD', 'GAS', 'ANT', 'ALO', 'LEC', 'STR',
       'TSU', 'ALB', 'HUL', 'LAW', 'OCO', 'COL', 'HAM', 'SAI', 'RUS',
       'PIA', 'BEA'], dtype=object)

In [8]:
track2["LapTime"].min()

'0 days 00:01:20.055000'

In [9]:
#Assign Rank for each entry point
track2["RK"] = track2.groupby(by=["Driver"])["LapTime"].rank(method="dense", ascending=True)
track2["Total_RK"] = track2["LapTime"].rank(method="dense", ascending=True)
track2["RK"]= track2["RK"].astype(int)

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_37839/2425921597.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track2["RK"] = track2.groupby(by=["Driver"])["LapTime"].rank(method="dense", ascending=True)
/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_37839/2425921597.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track2["Total_RK"] = track2["LapTime"].rank(method="dense", ascending=True)
/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_37839/2425921597.py:4: Setti

In [10]:
track_fastest=track2

In [11]:
pole_lap = track_fastest["LapTime"].min()
track_fastest['LapTimeDelta']= pd.to_timedelta(track_fastest["LapTime"]) - pd.to_timedelta(pole_lap)
track_fastest['LapTimeDelta']=track_fastest['LapTimeDelta'].astype(str)

for index, row in track_fastest.iterrows():
  if track_fastest.loc[index, 'LapTimeDelta']=='0 days 00:00:00':
    track_fastest.loc[index, 'LapTimeDelta']='0 days 00:00:00.01'
track_fastest['LapTimeDelta']=pd.to_timedelta(track_fastest['LapTimeDelta'])
track_fastest

track_fastest['LapTimeDelta'] = track_fastest['LapTimeDelta'] + pd.to_datetime('1970/01/01')

track_fastest['LapTimeDelta2']= pd.to_timedelta(track_fastest["LapTime"]) - pd.to_timedelta(pole_lap)
track_fastest['LapTimeDelta2']=track_fastest['LapTimeDelta2'].astype(str)

track_fastest["Driver_Lap"]=track_fastest["Driver"].astype(str)+ " " +"(Lap "+track_fastest["LapNumber"].astype(str)+")"


track_fastest["LapTimeDelta2"]=track_fastest['LapTimeDelta2'].str.split('days ').str[1]

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_37839/3793119071.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track_fastest['LapTimeDelta']= pd.to_timedelta(track_fastest["LapTime"]) - pd.to_timedelta(pole_lap)
/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_37839/3793119071.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track_fastest['LapTimeDelta']=track_fastest['LapTimeDelta'].astype(str)
/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_37839/3793119071.py:8: S

In [12]:
track_fastest_best_by_driver=track2[track2['RK']==1]

In [13]:
fig=px.bar(
    track_fastest_best_by_driver,
    y="Driver",
    x='LapTimeDelta',
    text='LapTimeDelta2',
    orientation='h',
    color='Driver',
    template="presentation",
    hover_data=['Team', 'Year', 'GP','LapTime', 'LapNumber','Sector1Time', 'Sector2Time', 'Sector3Time',
       'Compound', 'TyreLife', 'FreshTyre'],
    title="<b>Sprint Qualifying Delta vs. Fastest Lap for the {} {}</b>".format(year,gp),
    height=800, 
    width=1200,

color_discrete_map={
                 "VER": "#3671C6",
                 "TSU": "#3671C6",
                 "LEC": "#E80020",
                 "HAM": "#E80020",
                 "NOR": "#FF8000",
                 "PIA": "#FF8000",
                 "RUS": "#27F4D2",
                 "ANT": "#27F4D2",
                 "GAS": "#0093CC",
                 "DOO": "#0093CC",
                 "COL": "#0093CC",
                 "ALO": "#229971",
                 "STR": "#229971",
                 "SAI": "#64C4FF",
                 "ALB": "#64C4FF",
                 "HUL": "#52e252",
                 "BOR": "#52e252",
                 "LAW": "#6692FF",
                 "HAD": "#6692FF",
                 "OCO": "#B6BABD",
                 "BEA": "#B6BABD"
                 }
)
fig.update_layout(
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),

    xaxis_title="<b>Delta</b>",
    yaxis_title="<b>Driver</b>",
    title_font_family="<b>PT Sans Narrow</b>",

)

fig.update_layout(xaxis_tickformat='%H:%M:%S.%f')

fig.update_traces(marker_line_width=1,marker_line_color="BLACK")

fig.update_layout(yaxis={'categoryorder':'total descending'})

fig.update_traces(textposition='auto')

fig.update_layout(
    title_x=0.5,
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow",
    margin=dict(l=60, r=5, t=35, b=60),
)
fig.show()

In [14]:
fig.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Qualy Delta/{}/{} {} Sprint Qualifying Delta.html".format(year,year,gp),full_html=False, include_plotlyjs='cdn')

In [5]:
#2024
race_multi=['Austrian Grand Prix','Miami Grand Prix','Chinese Grand Prix']
#2023
# race_multi=['Austrian Grand Prix','Azerbaijan Grand Prix','Belgian Grand Prix','Qatar Grand Prix','São Paulo Grand Prix','United States Grand Prix']

year_multi=2024

for i in race_multi:
    track2=track3[(track3["GP"]==i)&(track3["Year"]==year_multi)]
    year=track2['Year'].iloc[0]
    gp=track2['GP'].iloc[0]
    Drivers=(track2['Driver'].unique())
    track2["LapTime"].min()
    #Assign Rank for each entry point
    track2["RK"] = track2.groupby(by=["Driver"])["LapTime"].rank(method="dense", ascending=True)
    track2["RK"]= track2["RK"].astype(int)
    track_fastest=track2[track2['RK']==1]
    pole_lap = track_fastest["LapTime"].min()
    track_fastest['LapTimeDelta']= pd.to_timedelta(track_fastest["LapTime"]) - pd.to_timedelta(pole_lap)
    track_fastest['LapTimeDelta']=track_fastest['LapTimeDelta'].astype(str)

    for index, row in track_fastest.iterrows():
     if track_fastest.loc[index, 'LapTimeDelta']=='0 days 00:00:00':
        track_fastest.loc[index, 'LapTimeDelta']='0 days 00:00:00.01'
    track_fastest['LapTimeDelta']=pd.to_timedelta(track_fastest['LapTimeDelta'])
    track_fastest

    track_fastest['LapTimeDelta'] = track_fastest['LapTimeDelta'] + pd.to_datetime('1970/01/01')

    track_fastest['LapTimeDelta2']= pd.to_timedelta(track_fastest["LapTime"]) - pd.to_timedelta(pole_lap)
    track_fastest['LapTimeDelta2']=track_fastest['LapTimeDelta2'].astype(str)

    track_fastest["LapTimeDelta2"]=track_fastest['LapTimeDelta2'].str.split('days ').str[1]
    fig=px.bar(
        track_fastest,
        y="Driver",
        x='LapTimeDelta',
        text='LapTimeDelta2',
        orientation='h',
        color='Driver',
        template="presentation",
        hover_data=['Team', 'Year', 'GP','LapTime', 'LapNumber','Sector1Time', 'Sector2Time', 'Sector3Time',
        'Compound', 'TyreLife', 'FreshTyre'],
        title="<b>Qualifying Delta vs. Fastest Lap for the {} {}</b>".format(year_multi,i),
        height=800, 
        width=1200,

    color_discrete_map={
                 "ALB": "#64C4FF",
                 "ALO": "#229971",
                 "BOT": "#52e252",
                 "COL": "#37BEDD",
                 "GAS": "#0093cc",
                 "HAM": "#27F4D2",
                 "HUL": "#B6BABD",
                 "LEC": "#E80020",
                 "MAG": "#B6BABD",
                 "NOR": "#FF8000",
                 "OCO": "#0093cc",
                 "PER": "#3671C6",
                 "PIA": "#FF8000",
                 "RIC": "#6692FF",
                 "RUS": "#27F4D2",
                 "SAI": "#E80020",
                 "SAR": "#37BEDD",
                 "STR": "#229971",
                 "TSU": "#6692FF",
                 "VER": "#3671C6",
                 "ZHO": "#52e252"
                    }
    )
    fig.update_layout(
        yaxis = dict(tickfont = dict(size=20)),
        xaxis = dict(tickfont = dict(size=20)),

        xaxis_title="<b>Delta</b>",
        yaxis_title="<b>Driver</b>",
        title_font_family="<b>PT Sans Narrow</b>",

    )

    fig.update_layout(xaxis_tickformat='%H:%M:%S.%f')

    fig.update_traces(marker_line_width=1,marker_line_color="BLACK")

    fig.update_layout(yaxis={'categoryorder':'total descending'})

    fig.update_traces(textposition='auto')

    fig.update_layout(
        title_x=0.5,
        hoverlabel=dict(
            bgcolor="white",
            font_size=16,
            font_family="PT Sans Narrow"
        ),
        yaxis = dict(tickfont = dict(size=20)),
        xaxis = dict(tickfont = dict(size=20)),
        font=dict(
            family="PT Sans Narrow",
            size=14,
            color="Black"
        ),
        title_font_family="PT Sans Narrow",
        margin=dict(l=60, r=5, t=35, b=60),
    )
    fig.write_html("/Users/rdesh723/statpulse-html/plots/Sprint Qualy Delta/{}/{} {} Sprint Qualifying Delta.html".format(year_multi,year_multi,i),full_html=False, include_plotlyjs='cdn')

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_6347/420432327.py:15: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_6347/420432327.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_6347/420432327.py:19: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docu